# Measure selection and read latency

Measure Paper greedy, Tiles, and Top-$R$ using the same captured importance,
requested row budgets, and direct-I/O reader.

Inputs are the profile from [01](../01/01_01_measure_chunk_latency.ipynb),
traces from [02](../02/02_01_capture_traces.ipynb), and settings from
[03_01](03_01_tune_paper.ipynb). Run from this directory on the target SSD;
see the [setup instructions](../../readme.md) for dependencies.

In [22]:
from pathlib  import Path
from tempfile import TemporaryDirectory
from time     import perf_counter_ns

import json
import os
import platform
import subprocess
import sys
import torch

import numpy  as np
import pandas as pd

from torch.utils.cpp_extension import load

In [23]:
UPSTREAM = Path("../../vlm-flash")

sys.path.insert(0, str((UPSTREAM / "src").resolve()))
sys.path.insert(0, str((UPSTREAM / "scripts").resolve()))

import vlmflash
from vlmflash._native import native
from _lock            import exclusive

## 1. Inputs and settings

Take up to 32 evenly spaced traces per shape in archive order.
For requested fraction $f$, the row budget is $R=\max(1,\lfloor fN\rfloor)$.
Tiles select exactly $R$ rows; Paper may select fewer.
At $R=N$, all methods bypass selection.

In [24]:
PROFILE = Path("../01/chunk_latency_measurements.csv")
TRACE   = Path("../02/activation_traces.json")
TUNING  = Path("paper_tuning.csv")

MAX_TRACES_PER_SHAPE = 32  # 0 uses all archived traces.

In [25]:
archive = json.loads(TRACE.read_text())

weight_bytes = archive["weight_bytes"]
shapes       = sorted({(trace["n"], trace["d"]) for trace in archive["traces"]})

traces = []

for shape in shapes:
    group = []

    for trace in archive["traces"]:
        if (trace["n"], trace["d"]) == shape:
            group.append(trace)

    count   = min(len(group), MAX_TRACES_PER_SHAPE or len(group))
    indices = np.linspace(0, len(group) - 1, count, dtype=int)

    for index in indices:
        traces.append(group[index])

In [26]:
tuning = pd.read_csv(TUNING)
tuning = tuning.set_index(["n", "d"], verify_integrity=True)

assert set(shapes) <= set(tuning.index), "Run 03_01 for the missing projection shapes."

In [27]:
measurements = pd.read_csv(PROFILE)

latencies = (
    measurements.groupby("size_kib")["latency_ms"]
    .median()
    .sort_index()
)

assert np.isfinite(latencies).all() and (latencies > 0).all()

tile_reference_kib = float((latencies / latencies.index).idxmin())
max_profiled_kib   = float(latencies.index.max())
latency_table      = vlmflash.LatencyTable(latencies.to_dict())

## 2. Selectors

Paper ranks windows by importance per estimated read cost on the GPU, then
greedily accepts disjoint windows on the CPU. Tiles ranks full tiles on the CPU,
fills the remaining budget, and compares offsets $0$ and $\lfloor B/2\rfloor$.

Compile Paper's native extension and [selectors.cpp](selectors.cpp) before timing.

In [28]:
os.environ.setdefault("MAX_JOBS", "1")
torch.set_num_threads(1)

assert torch.cuda.is_available(), "Paper's measured GPU sort requires CUDA."

In [29]:
reader = native()
assert reader is not None, "Build the upstream native extension first."

In [30]:
selectors = load(
    "vlmflash_selection03_notebook",
    [str(Path("selectors.cpp").resolve())],
    extra_cflags = ["-O3", "-std=c++17"],
    verbose      = True,
)

## 3. Candidate geometry

Let $z_*$ minimize median measured cost per KiB and $w$ be KiB per row.
Tiles use $s=z_*/w$, its coarse multiples $1,1.25,1.5,1.75,2$, and successive
halvings down to one row. Keep each integer width once. Paper uses the tuned windows.

The profile lookup supplies estimated read costs for selection. Estimated mask
cost sums these costs over maximal selected runs.

In [31]:
COARSE_SCALES = [1, 1.25, 1.5, 1.75, 2]
PAPER         = "Paper greedy (GPU sort)"

In [32]:
def tile_grid(s):

# 1. COARSE TILES
    scales = []

    for scale in COARSE_SCALES:
        scales.append((f"{scale:g}s", float(scale)))

# 2. FINER TILES
    levels = max(0, int(np.ceil(np.log2(max(1, s)))))

    for j in range(1, levels + 1):
        scales.append((f"s/2^{j}", 2.0 ** -j))

# 3. DISTINCT WIDTHS
    seen = set()
    grid = {}

    # Keep the first label for each integer width.
    for label, scale in scales:
        width = max(1, int(np.ceil(scale * s)))
        if width in seen:
            continue

        name = "Top-R" if width == 1 else f"Tiles ({label})"
        grid[name] = (scale, width)
        seen.add(width)

    return grid

In [33]:
def paper_windows(n, w, tuned):

# 1. SINGLE-ROW FALLBACK
    if max_profiled_kib < 2 * w:
        return torch.tensor([1]), 1

# 2. ROW GEOMETRY
    start  = max(1, int(tuned.start_kb / w))
    stride = max(1, int(tuned.jump_cap_kb / w))
    stop   = min(n + 1, int((max_profiled_kib + 1) / w))

# 3. CANDIDATE WINDOWS
    rows = list(range(start, stop, start))
    assert rows, "Paper window range is empty."

    return torch.tensor(rows), stride

In [34]:
def prepare(trace):

# 1. IMPORTANCE
    values = torch.tensor(trace["values"], dtype=torch.float32)
    assert values.ndim == 1 and torch.isfinite(values).all() and (values >= 0).all()

# 2. READ COST
    n       = trace["n"]
    row_kib = trace["row_bytes"] / 1024
    costs   = [0.0]

    for r in range(1, n + 1):
        costs.append(latency_table.read_ms(row_kib * r))

    costs = torch.tensor(costs, dtype=torch.float64)

# 3. PAPER
    d     = trace["d"]
    tuned = tuning.loc[(n, d)]

    sizes, stride = paper_windows(n, row_kib, tuned)
    delays        = costs[sizes]

    methods = {
        PAPER: lambda R: reader.select_chunks(
            values, 
            R, 
            sizes, 
            delays, 
            stride, 
            True,
        )[0],
    }

# 4. TILES
    grid = tile_grid(tile_reference_kib / row_kib)

    for name, (scale, width) in grid.items():
        methods[name] = lambda R, B=width: selectors.tiles(
            values,
            R,
            costs,
            B,
        )[0]

    return values, costs, methods, grid

## 4. Cases

A case is one trace, requested budget, and method. Record its selected mask as
maximal `[start, end)` intervals.

For captured mean absolute activations $v_i$, retained importance is
$100\sum_i v_iM_i/\sum_i v_i$ and budget fill is $100\|M\|_1/R$.

In [35]:
def chunks(mask):
    """Return selected chunks as [start, end) intervals."""

    mask   = mask.numpy().astype(bool)
    padded = np.concatenate(([0], mask, [0]))
    change = np.diff(padded)

    starts = np.flatnonzero(change ==  1) # 0 -> 1
    ends   = np.flatnonzero(change == -1) # 1 -> 0

    return np.column_stack((starts, ends))

In [36]:
def describe(mask, values, R, costs, method):

# 1. SELECTED ROWS
    selected = int(mask.sum())

    if method == PAPER:
        assert selected <= R
    else:
        assert selected == R

# 2. RETAINED IMPORTANCE
    total    = float(values.double().sum())
    retained = np.nan

    if total:
        importance = float(values[mask.bool()].double().sum())
        retained   = 100 * importance / total

# 3. READ COST
    spans   = chunks(mask)
    lengths = np.diff(spans).ravel()
    latency = float(costs[lengths].sum())

# 4. RESULT
    return {
        "R_selected"    : selected,
        "fill_pct"      : 100 * selected / R,
        "retained_pct"  : retained,
        "modeled_io_ms" : latency,
        "runs"          : len(spans),
        "intervals"     : json.dumps(spans.tolist()),
    }

## 5. Timing

`selection_ms` covers selection, including Paper's GPU sort and score transfers.
`io_ms` covers the native read/thread interval. `wall_ms` runs from prepared CPU
importance through completion of the CPU read, including read preparation.
GPU weight transfer and model execution are outside this interval.

The reader uses six threads and `O_DIRECT`, splitting requests at 768 KiB.
Validation follows timing.

In [37]:
def timed_read(select, R, expected, blob, row_bytes):
    full = R == expected.numel()

# 1. SELECTION
    begin        = perf_counter_ns()
    mask         = expected if full else select(R)
    after_select = perf_counter_ns()

# 2. READ
    raw, io_us, _, direct = reader.read_rows(
        blob, 
        mask, 
        row_bytes, 
        "cpu", 
        THREADS, 
        MAX_READ_KIB * 1024, 
        True,
    )
    end = perf_counter_ns()

# 3. VALIDATION (OUTSIDE TIMING)
    assert direct, "O_DIRECT unavailable; refusing page-cache timings."
    assert torch.equal(mask, expected)
    assert raw.shape == (int(expected.sum()), row_bytes)

# 4. RESULT
    return {
        "selection_ms" : (after_select - begin) / 1e6,
        "io_ms"        : io_us / 1000,
        "wall_ms"      : (end - begin) / 1e6,
    }

In [38]:
FRACTIONS = np.arange(3, 11) / 10

WARMUP  = 3
REPEATS = 30

THREADS      = 6
MAX_READ_KIB = 768

In [39]:
def measure_trace(trace, blob):
    values, costs, methods, grid = prepare(trace)

    n     = len(values)
    dense = torch.ones(n, dtype=torch.uint8)

    for fraction in FRACTIONS:
        R       = max(1, int(round(float(fraction), 6) * n))
        checked = {}

        # Selections
        for name, select in methods.items():
            mask         = dense if R == n else select(R)
            scale, width = grid.get(name, (np.nan, np.nan))

            if width == 1:
                order = torch.argsort(values, descending=True, stable=True)
                top   = torch.zeros(n, dtype=torch.uint8)
                top[order[:R]] = 1

                assert torch.equal(mask, top), "One-row tiles must equal stable Top-R."

            case_id = len(case_records)

            case_records.append({
                # Trace
                "case_id"  : case_id,
                "trace_id" : trace["trace_id"],
                "n"        : n,
                "d"        : trace["d"],

                # Requested budget
                "fraction"  : float(fraction),
                "R_nominal" : R,

                # Selection method
                "tile_scale" : scale,
                "tile_rows"  : width,
                "tile_kib"   : width * trace["row_bytes"] / 1024,
                "method"     : name,

                # Selected mask
                **describe(mask, values, R, costs, name),
            })

            checked[name] = (case_id, mask.clone())

        # Repeated measurements
        for repeat in range(-WARMUP, REPEATS):
            for name in rng.permutation(list(methods)):
                case_id, mask = checked[name]
                timing = timed_read(
                    methods[name], R, mask, blob, trace["row_bytes"],
                )

                if repeat >= 0:  # Exclude warmup rounds.
                    timing_records.append({
                        "case_id" : case_id,
                        "repeat"  : repeat,
                        **timing,
                    })

## 6. Measurement

Allocate and flush a random file sized for projection rows. For each trace and
budget, randomize method order in every round. Discard three warmup rounds and
record 30 repetitions. Hold the experiment lock throughout the run.

In [40]:
def write_blob(path, size):
    with open(path, "wb") as stream:
        os.posix_fallocate(stream.fileno(), 0, size)

        for begin in range(0, size, 2**20):
            count = min(2**20, size - begin)
            stream.write(os.urandom(count))

        stream.flush()
        os.fsync(stream.fileno())

In [41]:
rng = np.random.default_rng(0)

case_records   = []
timing_records = []

with exclusive("experiment03"), TemporaryDirectory(dir=Path.cwd()) as work:
    blob = str(Path(work) / "rows.dat")
    size = max(trace["n"] * trace["row_bytes"] for trace in traces)
    write_blob(blob, size)

    for index, trace in enumerate(traces, 1):
        measure_trace(trace, blob)

        if index % 16 == 0 or index == len(traces):
            print(f"{index}/{len(traces)} traces measured", flush=True)

16/504 traces measured
32/504 traces measured
48/504 traces measured
64/504 traces measured


KeyboardInterrupt: 

## 7. Output

`cases.csv` stores masks and metadata; `timings.csv` stores repeated measurements.
The files join on `case_id`. `environment.json` records the device settings and
tuning used for this run.

In [ ]:
cases   = pd.DataFrame(case_records)
timings = pd.DataFrame(timing_records)

assert cases.case_id.is_unique
assert not timings.duplicated(["case_id", "repeat"]).any()
assert timings.groupby("case_id").size().eq(REPEATS).all()

In [ ]:
paper_tuning = []

for n, d in shapes:
    setting = tuning.loc[(n, d)]

    best = {
        "start_kb"    : int(setting.start_kb),
        "jump_cap_kb" : int(setting.jump_cap_kb),
        "overhead_ms" : float(setting.overhead_ms),
    }

    paper_tuning.append({
        "n"             : n,
        "d"             : d,
        "best"          : best,
        "num_feasible"  : int(setting.num_feasible),
        "num_evaluated" : int(setting.num_evaluated),
    })

In [ ]:
upstream_commit = subprocess.check_output(
    ["git", "-C", str(UPSTREAM), "rev-parse", "HEAD"],
    text = True,
).strip()

mount = subprocess.check_output(
    ["findmnt", "-T", str(Path.cwd()), "-n", "-o", "SOURCE,FSTYPE,TARGET"],
    text = True,
).strip()

In [ ]:
environment = {
    # Device and implementation
    "platform"        : platform.platform(),
    "torch"           : torch.__version__,
    "cuda_available"  : torch.cuda.is_available(),
    "upstream_commit" : upstream_commit,
    "mount"           : mount,

    # Selection and reader
    "paper_sort_devices" : ["cuda"],
    "paper_tuning"       : paper_tuning,
    "torch_threads"      : torch.get_num_threads(),
    "reader_threads"     : THREADS,
    "direct_io"          : True,
    "destination"        : "cpu",
    "max_read_kib"       : MAX_READ_KIB,

    # Candidate geometry
    "tile_boundary_kib" : tile_reference_kib,
    "coarse_scales"     : COARSE_SCALES,
    "dtype_bytes"       : weight_bytes,
    "tile_grid"         : "halving_to_one_row; unique widths per shape",
    "tile_offsets"      : "0, floor(B/2)",
    "paper_ceiling_kib" : max_profiled_kib,

    # Measurement
    "traces"      : len(traces),
    "warmup"      : WARMUP,
    "repeats"     : REPEATS,
    "fractions"   : FRACTIONS.tolist(),
    "measurement" : "prepared CPU importance through selected bytes on CPU; no model compute",

    # Output
    "cases_file"   : "cases.csv",
    "timings_file" : "timings.csv",
    "cases"        : len(cases),
    "samples"      : len(timings),
}

In [ ]:
cases.to_csv("cases.csv", index=False)
timings.to_csv("timings.csv", index=False)

Path("environment.json").write_text(json.dumps(environment, indent=2) + "\n")
print(f"Saved {len(cases):,} cases and {len(timings):,} repeat samples.")

Saved 14,080 cases and 422,400 repeat samples.
